# Social Network Analysis with NetworkX
## Revised practical: Zachary's Karate Club

**Purpose.** Explore a real social network, compare it with Erdős–Rényi random graphs, implement triangle counting, and investigate community structure.

### Learning outcomes
By the end of this practical, you should be able to:
1. Load, inspect, and visualize an undirected social network.
2. Explain basic network statistics and centrality measures.
3. Compare an empirical network with an ensemble of random graph realizations.
4. Implement and validate a triangle-counting algorithm.
5. Detect communities and critically compare them with known group labels.

### Working conventions
- Keep the original graph `G` unchanged; use `G.copy()` if you need to modify it.
- Set random seeds for reproducibility.
- Run cells from top to bottom.
- A number or plot alone is not a complete answer: explain what the result means.

### Documentation
- [NetworkX](https://networkx.org/documentation/stable/)
- [Matplotlib](https://matplotlib.org/stable/)

## Section 0 — Setup and first look

Install `networkx`, `matplotlib`, and `numpy` if necessary.

The Karate Club graph contains 34 people (nodes) and 78 social ties (edges). The node attribute `club` records the faction associated with each person after the split.

**Tasks**
1. Import the libraries and load the graph.
2. Print the number of nodes, edges, and average degree.
3. Inspect the `club` node attribute.
4. Draw the network, coloring nodes by recorded club membership.
5. Describe visible clusters or bridge-like nodes.

**Predict first:** Do you expect the people with the most connections to be the most influential in every sense? Explain before calculating centrality.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

RANDOM_SEED = 42
NUM_TRIALS = 200

G = nx.karate_club_graph()
N = G.number_of_nodes()
E = G.number_of_edges()
avg_deg = 2 * E / N

print(f"Nodes: {N}")
print(f"Edges: {E}")
print(f"Average degree: {avg_deg:.3f}")
print("Club labels:", sorted({G.nodes[n]["club"] for n in G}))

In [ ]:
# TODO: Draw G with nodes colored by the 'club' attribute.
# Hint: create a color list in the same order as G.nodes().
pos = nx.spring_layout(G, seed=RANDOM_SEED)
# Your code here

## Section 1 — Distances and centrality

### 1.1 Distances
Calculate:
- **Diameter:** the largest shortest-path distance between any pair of nodes.
- **Average shortest path length:** the mean shortest-path distance over all node pairs.

The Karate Club graph is connected. For a disconnected graph, the usual average shortest path length is not defined over all pairs.

**Questions**
1. What are the diameter and average shortest path length?
2. Why is the average distance no larger than the diameter?
3. What does each measure tell you about communication or separation?

In [ ]:
assert nx.is_connected(G), "Expected the connected Karate Club graph."

# TODO: Calculate both measures.
diameter = None
avg_shortest_path = None

print("Diameter:", diameter)
print("Average shortest path length:", avg_shortest_path)

### 1.2 Compare three centrality measures

Calculate:
- **Degree:** number of direct neighbors.
- **PageRank:** recursive importance based on the importance of neighbors.
- **Betweenness centrality:** how often a node lies on shortest paths between other nodes.

**Tasks**
1. Calculate all three measures.
2. Print the five highest-ranked nodes for each.
3. Plot degree against PageRank and label points by node.
4. Identify nodes whose rankings differ across measures.

**Questions**
- Are the top-ranked nodes the same?
- What structural role does betweenness capture that degree does not?
- Why should “most central” be qualified by the measure being used?

In [ ]:
degrees = dict(G.degree())

# TODO: Calculate centrality dictionaries.
pagerank = None
betweenness = None

# TODO: Print the top five nodes for each measure.
# TODO: Plot degree vs. PageRank, labeling points by node.

## Section 2 — Comparing the empirical graph with Erdős–Rényi random graphs

In \(G(N,p)\), each possible edge is included independently with probability \(p\). Choose \(p\) so that the model's **expected** average degree matches the observed average degree:
\[
p = \frac{\langle k\rangle}{N-1}.
\]
This matches one property only; it does not guarantee that a random realization resembles the empirical graph in other respects.

### 2.1 One realization
1. Calculate `p`.
2. Generate one graph with the same `N` and `p`.
3. Plot it beside the Karate Club graph using the same layout method.
4. Compare visible structures. A single realization is not representative of the whole model.

In [ ]:
p = avg_deg / (N - 1)
G_rand = nx.gnp_random_graph(N, p, seed=RANDOM_SEED)

# TODO: Plot G and G_rand side by side.

### 2.2 Ensemble experiment

Generate at least 100 independent \(G(N,p)\) realizations. For each, record:
- Average shortest path length.
- Degree assortativity coefficient.
- Transitivity (global clustering coefficient).
- Whether the graph is connected.

**Important:** average shortest path length over all nodes is only defined for connected realizations. Record `NaN` for disconnected graphs and report the connected fraction. Optionally calculate path length on the largest connected component, but label it as a different quantity.

**Tasks**
1. Run at least 100 trials (200 suggested).
2. Store each realization's metric values.
3. Report mean and standard deviation. For path length, state that the summary is conditional on connected realizations.
4. Use boxplots and add the empirical Karate Club value as a reference line.
5. Compare the empirical values with the random ensemble.

**Questions**
- How variable are metrics across realizations?
- Does the empirical graph fall near the center or in the tail?
- Does matching expected average degree reproduce clustering and assortativity?
- What does the connectedness rate tell you?

In [ ]:
results = {
    "avg_path_length": [],
    "assortativity": [],
    "transitivity": [],
    "connected": []
}

rng = np.random.default_rng(RANDOM_SEED)
seeds = rng.integers(0, 2**32 - 1, size=NUM_TRIALS, dtype=np.uint32)

for seed in seeds:
    H = nx.gnp_random_graph(N, p, seed=int(seed))
    is_connected = nx.is_connected(H)
    results["connected"].append(is_connected)

    # TODO: Append average path length if connected, otherwise np.nan.
    # TODO: Append degree assortativity and transitivity.
    pass

# TODO: Summarize means and standard deviations.
# TODO: Plot distributions with empirical reference lines.

## Section 3 — Counting triangles from scratch

A **triangle** is a set of three distinct nodes that are all pairwise connected.

Implement a function returning `{node: triangle_count}`. Do not use `nx.triangles` or `nx.transitivity` inside your implementation.

**Tasks**
1. Implement `count_triangles_per_node(graph)`.
2. Validate against `nx.triangles(G)`.
3. Calculate the number of unique triangles. Each triangle contributes to three node counts, so divide their sum by three.
4. Calculate the expected number of triangles in \(G(N,p)\):
\[
\mathbb{E}[T] = \binom{N}{3}p^3.
\]
5. Explain why the expectation need not equal the count in one random realization.
6. Discuss the computational cost of your implementation.

In [ ]:
def count_triangles_per_node(graph):
    # Return {node: number of triangles containing node}.
    tri_counts = {u: 0 for u in graph.nodes()}
    # TODO: Implement triangle counting without nx.triangles.
    return tri_counts

my_triangles = count_triangles_per_node(G)

# TODO: Validate against nx.triangles(G).
# TODO: Calculate total unique triangles and E[T] for G(N,p).

## Section 4 — Community detection

The graph includes a `club` attribute recording the two factions. Compare these labels with communities detected from graph structure.

Use greedy modularity maximization:
```python
communities = nx.community.greedy_modularity_communities(G)
```

**Tasks**
1. Detect communities and report their number and sizes.
2. Calculate modularity.
3. Visualize the detected communities.
4. Compare the detected partition with recorded `club` labels. If you calculate an agreement score, explain what it measures.
5. Identify nodes where the assignments differ.

**Questions**
- Does the algorithm recover the two recorded factions?
- Why might graph-based communities differ from recorded labels?
- Does high modularity prove that communities are meaningful social groups?

An algorithmic partition is a model-based result, not ground truth.

In [ ]:
communities = nx.community.greedy_modularity_communities(G)
modularity = nx.community.modularity(G, communities)

print("Number of communities:", len(communities))
print("Community sizes:", sorted([len(c) for c in communities], reverse=True))
print("Modularity:", modularity)

# TODO: Visualize detected communities.
# TODO: Compare with the 'club' node attribute.

## Section 5 — Final synthesis

Write a conclusion (approximately 250–400 words) answering:
1. What do the centrality measures reveal about the network?
2. In which respects does the Erdős–Rényi ensemble differ from the empirical graph?
3. What did triangle counting reveal about local structure?
4. How well did community detection recover the recorded factions?
5. Why is no single statistic sufficient to characterize a social network?

### Submission checklist
- [ ] All code cells run without errors.
- [ ] Figures have titles and appropriate labels.
- [ ] Random seed and number of trials are reported.
- [ ] Random-graph variability and connectedness are discussed.
- [ ] Triangle implementation is validated.
- [ ] Community detection is compared with recorded labels.
- [ ] Written interpretations refer to evidence from results.